# Lab 11 challenge: worked solution

For the instructor. It copies the lab's copilot, `stream_turn`, `time_case` and `Ledger` from the lab notebook, so it
runs on its own. Model runs vary: compare counts and shapes, not wording or seconds.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "REASONING_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
REASONING_MODEL   (not set)
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


In [3]:
# Copied from the lab: the Ops API, the copilot, the timer, the case runner and the ledger.
import math, time, uuid
from statistics import median

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

import shared.ops_tools as ops_tools
from shared.copilot import search_policies
from shared.eval_cases import check, lookup, tool_calls
from shared.ops_api import start_ops_api
from shared.ops_tools import (READ_TOOLS, RepContext, copilot_prompt, file_refund_request, find_customer,
                              get_customer_invoices, get_invoice)
from shared.settings import chat_model

server = start_ops_api()
TOOLS = [search_policies, find_customer, get_customer_invoices, get_invoice, file_refund_request]
REP = RepContext(rep_id="rep-3", rep_name="Jane Peacock")
LIVE_API, DOWN_API = ops_tools.API, "http://127.0.0.1:9"

def build_copilot(prompt):
    approval = HumanInTheLoopMiddleware(
        interrupt_on={"file_refund_request": {"allowed_decisions": ["approve", "edit", "reject"]}})
    return create_agent(chat_model(settings), tools=TOOLS, system_prompt=prompt, middleware=[approval],
                        checkpointer=InMemorySaver(), context_schema=RepContext)

def new_thread():
    return {"configurable": {"thread_id": uuid.uuid4().hex}}

def stream_turn(agent, payload, cfg, t0):
    first_text = None
    for part in agent.stream(payload, cfg, context=REP, stream_mode="messages", version="v2"):
        chunk, meta = part["data"]
        if meta["langgraph_node"] == "model" and chunk.text and first_text is None:
            first_text = time.perf_counter() - t0
    return first_text

def time_case(agent, case, callbacks=()):
    cfg = {**new_thread(), "callbacks": list(callbacks)}
    ops_tools.API = DOWN_API if case.get("api_down") else LIVE_API
    reject = {"type": "reject", "message": case.get("decision", "Not approved by the reviewer. Nothing was filed.")}
    t0 = time.perf_counter()
    first_text = stream_turn(agent, {"messages": [("user", case["input"])]}, cfg, t0)
    while agent.get_state(cfg).interrupts:
        waiting = agent.get_state(cfg).interrupts[0].value["action_requests"]
        resumed = stream_turn(agent, Command(resume={"decisions": [reject] * len(waiting)}), cfg, t0)
        first_text = first_text or resumed
    total = time.perf_counter() - t0
    ops_tools.API = LIVE_API
    messages = agent.get_state(cfg).values["messages"]
    return {"first_text": first_text, "total": total, "model_calls": sum(1 for m in messages if m.type == "ai"),
            "tools": [name for name, _ in tool_calls(messages)], "problems": check(case, messages),
            "reply": messages[-1].text}

class Ledger(BaseCallbackHandler):
    def __init__(self):
        self.started, self.rows = {}, []

    def on_chat_model_start(self, serialized, messages, *, run_id, metadata=None, **kwargs):
        meta = metadata or {}
        self.started[run_id] = (meta.get("lc_source") or meta.get("langgraph_node"), time.perf_counter())

    def on_llm_end(self, response, *, run_id, **kwargs):
        source, start = self.started.pop(run_id)
        usage = response.generations[0][0].message.usage_metadata or {}
        self.rows.append({"source": source, "input": usage.get("input_tokens", 0),
                          "output": usage.get("output_tokens", 0)})

expired_link = {
    "id": "expired link", "type": "answerable",
    "input": "A customer's download link expired. Draft a short reply I can send.",
    "want": "Search the handbook and offer one new link that lasts 7 days.",
    "calls": {"search_policies": {}}, "never": ["file_refund_request"], "reply_has": ["7 days"],
}
TURNS = ["I'm handling Helena Holý today. How many orders has she placed?",
         "What does our policy say about a disc that arrived cracked?",
         "She would like a replacement. Is that allowed?",
         "Draft a two-sentence reply I can send her.",
         "Before I close: how many orders did she place?"]
copilot = build_copilot(copilot_prompt())

Ops API started at http://127.0.0.1:8765


## 1. Make the fast case pass, and time it again

One line added to the prompt, then *expired link* and *lookup* three times each with the old and the new copilot.

In [4]:
LINE = "Before you answer or draft a reply about a customer's request, call search_policies."
copilots = {"course prompt": copilot, "with the line": build_copilot(copilot_prompt() + "\n" + LINE)}

for name, agent in copilots.items():
    for case in (expired_link, lookup):
        runs = [time_case(agent, case) for _ in range(3)]
        passed = sum(1 for r in runs if not r["problems"])
        print(f"{name:14}{case['id']:14}passed {passed} of 3 | total median {median(r['total'] for r in runs):.2f} s"
              f" | calls {[r['model_calls'] for r in runs]} | tools {runs[0]['tools']}")

course prompt expired link  passed 0 of 3 | total median 1.49 s | calls [1, 1, 1] | tools []


course prompt lookup        passed 3 of 3 | total median 3.08 s | calls [3, 3, 3] | tools ['find_customer', 'get_customer_invoices']


with the line expired link  passed 3 of 3 | total median 2.44 s | calls [2, 2, 2] | tools ['search_policies']


with the line lookup        passed 3 of 3 | total median 2.65 s | calls [3, 3, 3] | tools ['find_customer', 'get_customer_invoices']


Ours: with the line, *expired link* called `search_policies` and passed 3 of 3 (2 model calls, median 2.44 s), against 0 of 3 at 1.49 s with the course prompt; *lookup* passed 3 of 3 at 2.65 to 3.08 s with both. The right answer cost about a second and one model call. Run all eight cases before keeping the line: Lab 10's Challenge 1 shows a prompt line can move other cases.

## 2. Optional: a token budget per thread, with a handover

The hook runs before every model call. It adds up `input_tokens` from the AI messages saved in the thread and, above
the budget, writes the handover itself and jumps to the end, so no model call is made.

In [5]:
from langchain.agents.middleware import SummarizationMiddleware, before_model
from langchain_core.messages import AIMessage

BUDGET = 6_000

@before_model(can_jump_to=["end"])
def token_budget(state, runtime):
    used = sum(m.usage_metadata["input_tokens"] for m in state["messages"] if m.type == "ai" and m.usage_metadata)
    if used > BUDGET:
        text = (f"This case has used {used:,} input tokens, over its budget of {BUDGET:,}, so I am stopping here. "
                "Please hand it to your team lead with this thread id; the conversation so far is saved.")
        return {"messages": [AIMessage(text)], "jump_to": "end"}
    return None

def run_thread(agent, thread_id):
    cfg = {"configurable": {"thread_id": thread_id}}
    spent = 0
    for number, text in enumerate(TURNS, start=1):
        ledger = Ledger()
        result = agent.invoke({"messages": [("user", text)]}, {**cfg, "callbacks": [ledger]}, context=REP)
        spent += sum(r["input"] for r in ledger.rows)
        sources = [r["source"] for r in ledger.rows]
        print(f"turn {number}: calls {sources} | ledger input so far {spent:6,} -> {result['messages'][-1].text[:100]}")

budgeted = create_agent(chat_model(settings), tools=READ_TOOLS, system_prompt=copilot_prompt(),
                        context_schema=RepContext, middleware=[token_budget], checkpointer=InMemorySaver())
run_thread(budgeted, "budget-only")

turn 1: calls ['model', 'model', 'model'] | ledger input so far  2,659 -> Helena Holý has placed 7 orders totaling $49.62.


turn 2: calls ['model', 'model'] | ledger input so far  5,221 -> If a disc arrives cracked, it will be replaced at no charge. The customer should photograph the dama


turn 3: calls ['model'] | ledger input so far  6,764 -> Yes, a replacement for a cracked disc is allowed according to our policy [policy_damaged_media.md]. 
turn 4: calls [] | ledger input so far  6,764 -> This case has used 6,764 input tokens, over its budget of 6,000, so I am stopping here. Please hand 
turn 5: calls [] | ledger input so far  6,764 -> This case has used 6,764 input tokens, over its budget of 6,000, so I am stopping here. Please hand 


Ours (`gpt-4o-mini`, 13 Oct 2026): turns 1 to 3 answered with 3, 2 and 1 calls; before turn 4's first call the thread had used 6,764 input tokens, so turns 4 and 5 got the handover with no model call.

Now with the summariser in front of the budget.

In [6]:
with_summary = create_agent(
    chat_model(settings), tools=READ_TOOLS, system_prompt=copilot_prompt(), context_schema=RepContext,
    middleware=[SummarizationMiddleware(model=chat_model(settings), trigger=("tokens", 1500), keep=("messages", 4)),
                token_budget],
    checkpointer=InMemorySaver())
run_thread(with_summary, "budget-and-summary")

turn 1: calls ['model', 'model', 'model'] | ledger input so far  2,659 -> Helena Holý has placed a total of 7 orders. Suggested reply: "Helena, you've placed a total of 7 ord


turn 2: calls ['model', 'model'] | ledger input so far  5,253 -> If a disc arrives cracked, we will replace it at no charge. The customer should photograph the damag


turn 3: calls ['summarization', 'model'] | ledger input so far  7,593 -> Yes, a replacement is allowed if the disc arrived cracked. The customer should send a photo of the d


turn 4: calls ['model'] | ledger input so far  9,072 -> "Thank you for reaching out. Please send us a photo of the cracked disc, and we’ll arrange a replace


turn 5: calls ['summarization', 'model'] | ledger input so far 11,412 -> Helena Holý has placed a total of 7 orders.


Ours: the budget never fired. The summariser ran in turns 3 and 5 and replaced the older messages, AI messages included, so the sum the budget reads fell, while the ledger counted 11,412 input tokens over the five turns. A budget that reads the saved thread inherits Part D's blind spot: keep the running total from a callback, outside the messages the summariser rewrites.

### A budget that reads the ledger

The same handover, but the running total comes from a `Ledger` passed on every turn of the thread, so it counts every
model call, the summariser's included, and the summary cannot lower it. `ledger_budget` builds the hook around one
ledger.

In [7]:
def ledger_budget(ledger, budget=BUDGET):
    # The budget hook again, reading the ledger's input total instead of the saved messages.
    @before_model(can_jump_to=["end"])
    def budget_from_ledger(state, runtime):
        used = sum(row["input"] for row in ledger.rows)
        if used > budget:
            text = (f"This case has used {used:,} input tokens, over its budget of {budget:,}, so I am stopping here. "
                    "Please hand it to your team lead with this thread id; the conversation so far is saved.")
            return {"messages": [AIMessage(text)], "jump_to": "end"}
        return None
    return budget_from_ledger

case_ledger = Ledger()                 # one ledger for the whole thread
ledger_budgeted = create_agent(
    chat_model(settings), tools=READ_TOOLS, system_prompt=copilot_prompt(), context_schema=RepContext,
    middleware=[SummarizationMiddleware(model=chat_model(settings), trigger=("tokens", 1500), keep=("messages", 4)),
                ledger_budget(case_ledger)],
    checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "budget-from-ledger"}, "callbacks": [case_ledger]}
for number, text in enumerate(TURNS, start=1):
    before = len(case_ledger.rows)
    result = ledger_budgeted.invoke({"messages": [("user", text)]}, cfg, context=REP)
    sources = [row["source"] for row in case_ledger.rows[before:]]
    total = sum(row["input"] for row in case_ledger.rows)
    print(f"turn {number}: calls {sources} | ledger input so far {total:6,} -> {result['messages'][-1].text[:100]}")

turn 1: calls ['model', 'model', 'model'] | ledger input so far  2,659 -> Helena Holý has placed 7 orders, totaling $49.62.


turn 2: calls ['model', 'model'] | ledger input so far  5,223 -> If a disc arrives cracked, we will replace it at no charge. The customer should photograph the damag


turn 3: calls ['summarization'] | ledger input so far  6,152 -> This case has used 6,152 input tokens, over its budget of 6,000, so I am stopping here. Please hand 
turn 4: calls [] | ledger input so far  6,152 -> This case has used 6,152 input tokens, over its budget of 6,000, so I am stopping here. Please hand 
turn 5: calls [] | ledger input so far  6,152 -> This case has used 6,152 input tokens, over its budget of 6,000, so I am stopping here. Please hand 


Ours: the ledger budget fired in turn 3. The summariser's own call brought the ledger to 6,152 input tokens, over the
6,000 budget, so the hook stopped the turn before its model call, and turns 4 and 5 got the handover with no call at
all. It fired a turn earlier than the budget alone did (6,764 before turn 4), because the ledger also counts the
summariser. One limit: this ledger lives in the notebook's memory. A service keeps the running total per thread in its
database, keyed by the thread id, so a restart or a second server sees the same total.

## 3. Optional: how steady is the spread?

In [8]:
def pct(values, p):
    # Nearest-rank percentile: sort, then take the value at position ceil(p/100 * n).
    ordered = sorted(values)
    return ordered[max(0, math.ceil(p / 100 * len(ordered)) - 1)]

totals = [time_case(copilot, lookup)["total"] for _ in range(10)]
print("sorted:", [round(t, 2) for t in sorted(totals)])
print(f"p50 {pct(totals, 50):.2f} s | p95 {pct(totals, 95):.2f} s | slowest {max(totals):.2f} s")

sorted: [2.75, 2.8, 2.88, 2.96, 3.1, 3.19, 3.54, 3.85, 4.15, 4.32]
p50 3.10 s | p95 4.32 s | slowest 4.32 s


Ours: ten runs from 2.75 to 4.32 s; p50 3.10 s and p95 4.32 s, which with ten values is the slowest run. More runs make p95 steadier; ten show the shape, not a target.

In [9]:
if server is not None:          # None: another notebook runs the Ops API; stop it there
    server.should_exit = True